# 🫁 TB IGRA Longitudinal Study — Data Analysis
### Cohort Exploration, Feature Engineering & Visualization

**Author:** Saloni Prasad  
**Dataset:** `case_study.csv` — 149 subjects × 23 columns — 24-month longitudinal TB cohort  
**Tools:** Python · Pandas · NumPy · Matplotlib · Seaborn  

---

### Notebook Structure
1. Data Loading & Initial Exploration
2. Cohort Distribution Overview
3. Longitudinal IGRA Trajectory Analysis
4. Key Risk Factor Insights
5. Feature Engineering (BMI Categories + IFN-Gamma Delta)
6. Summary Statistics on Engineered Features
7. Visualizations

> **Note:** Place your dataset at `../data/raw/case_study.csv` before running.

---
## 1. Data Loading & Initial Exploration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

# Load dataset
df = pd.read_csv('../data/raw/case_study.csv')

# Strip whitespace from column names (important for columns like 'Month- 18 IGRA')
df.columns = df.columns.str.strip()

print(f'Dataset shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head()

---
## 2. Cohort Distribution Overview

In [ ]:
# Print distribution of key categorical variables
print('=== COHORT DISTRIBUTION ===')
print(f"\nOutcome Groups:      {df['OUTCOME'].value_counts().to_dict()}")
print(f"TB Status:           {df['TB STATUS'].value_counts().to_dict()}")
print(f"Smoking:             {df['SMOKING'].value_counts().to_dict()}")
print(f"Diabetes:            {df['DIABETES STATUS'].value_counts().to_dict()}")
print(f"BCG Vaccination:     {df['BCG VACCINATION'].value_counts().to_dict()}")
print(f"Region:              {df['REGION'].value_counts().to_dict()}")

---
## 3. Longitudinal IGRA Trajectory Analysis

In [ ]:
# Mode IGRA status across all 5 timepoints per outcome group
# Reveals the typical pattern for each clinical trajectory
igra_cols = ['Baseline IGRA', 'Month-6 IGRA', 'Month-12 IGRA', 'Month- 18 IGRA', 'Month24 IGRA']

print('=== IGRA STATUS MODE BY OUTCOME × TIMEPOINT ===')
print(df.groupby('OUTCOME')[igra_cols].agg(lambda x: x.mode()[0] if not x.empty else ''))

# Cross-tabulation: Outcome vs TB Status
print('\n=== TB STATUS vs OUTCOME ===')
print(pd.crosstab(df['OUTCOME'], df['TB STATUS']))

# Mean biomarker and clinical metrics per outcome group
print('\n=== MEAN IFN-GAMMA, BMI & AGE BY OUTCOME ===')
print(df.groupby('OUTCOME')[['IFN-GAMMA-UNS', 'IFN-GAMMA-C+E', 'BMI', 'AGE']].mean().round(2))

---
## 4. Key Risk Factor Insights

In [ ]:
# Compute percentage statistics for key findings
total = len(df)
progressors = df[df['OUTCOME'] == 'PROGRESSOR']

prog_diabetes = len(progressors[progressors['DIABETES STATUS'] == 'Yes'])
prog_smoking  = len(progressors[progressors['SMOKING'] == 'Yes'])
prog_bcg      = len(progressors[progressors['BCG VACCINATION'] == 'Yes'])
active_tb     = len(df[df['TB STATUS'] == 'Active'])

print('=' * 55)
print('          KEY RISK FACTOR INSIGHTS')
print('=' * 55)
print(f'Total Cohort Size          : {total}')
print(f'Active TB Cases            : {active_tb} ({active_tb/total*100:.1f}%)')
print(f'Progressor Group Size      : {len(progressors)}')
print()
print(f'Progressors with Diabetes  : {prog_diabetes}/{len(progressors)} ({prog_diabetes/len(progressors)*100:.1f}%)')
print(f'Progressors with Smoking   : {prog_smoking}/{len(progressors)} ({prog_smoking/len(progressors)*100:.1f}%)')
print(f'Progressors with BCG       : {prog_bcg}/{len(progressors)} ({prog_bcg/len(progressors)*100:.1f}%)')
print('=' * 55)
print()
print('KEY FINDING: 100% of PROGRESSORs had BOTH Diabetes AND Smoking.')
print('             14/15 Progressors developed Active TB (93.3%).')

---
## 5. Feature Engineering

In [ ]:
# --- Standardize gender casing ---
df['GENDER'] = df['GENDER'].str.upper()

# --- BMI Category Classification (WHO Standard) ---
def classify_bmi(bmi):
    if bmi < 18.5:
        return 'Underweight'
    elif 18.5 <= bmi < 25.0:
        return 'Normal'
    elif 25.0 <= bmi < 30.0:
        return 'Overweight'
    else:
        return 'Obese'

df['BMI_CATEGORY'] = df['BMI'].apply(classify_bmi)

# --- Biomarker Response Delta (Stimulated - Unstimulated IFN-Gamma) ---
# High IFN_GAMMA_DELTA = strong antigen-specific T-cell response
df['IFN_GAMMA_DELTA'] = df['IFN-GAMMA-C+E'] - df['IFN-GAMMA-UNS']

# --- Export processed dataset for Tableau dashboard ingestion ---
df.to_csv('../data/processed/processed_tb_cohort_analysis.csv', index=False)
print('Feature engineering complete.')
print('Processed dataset saved to: ../data/processed/processed_tb_cohort_analysis.csv')
print()
print('New features added:')
print('  BMI_CATEGORY     — WHO standard classification (Underweight/Normal/Overweight/Obese)')
print('  IFN_GAMMA_DELTA  — Stimulated minus Unstimulated IFN-Gamma (pg/mL)')

# Preview processed dataframe
df.head()

---
## 6. Summary Statistics on Engineered Features

In [ ]:
# Load the processed dataset
df_processed = pd.read_csv('../data/processed/processed_tb_cohort_analysis.csv')

# BMI Category distribution
print('=== BMI CATEGORY DISTRIBUTION ===')
print(df_processed['BMI_CATEGORY'].value_counts())
print()

# IFN-Gamma Delta statistics
print('=== IFN_GAMMA_DELTA STATISTICS ===')
print(df_processed['IFN_GAMMA_DELTA'].describe().round(3))

---
## 7. Visualizations

In [ ]:
# --- Visual 1: Distribution of BMI Categories ---
plt.figure(figsize=(8, 6))
sns.countplot(
    data=df_processed,
    x='BMI_CATEGORY',
    order=df_processed['BMI_CATEGORY'].value_counts().index,
    hue='BMI_CATEGORY',
    palette='viridis',
    legend=False
)
plt.title('Distribution of BMI Categories', fontsize=14, fontweight='bold', pad=12)
plt.xlabel('BMI Category', fontsize=12)
plt.ylabel('Count', fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.savefig('../visuals/bmi_category_distribution.png', bbox_inches='tight', dpi=150)
plt.show()
print('Chart saved: ../visuals/bmi_category_distribution.png')

In [ ]:
# --- Visual 2: Age vs IFN-Gamma Delta by BMI Category ---
# Reveals whether age and immune response delta vary across BMI groups
plt.figure(figsize=(10, 7))
sns.scatterplot(
    data=df_processed,
    x='AGE',
    y='IFN_GAMMA_DELTA',
    hue='BMI_CATEGORY',
    palette='viridis',
    s=100,
    alpha=0.7
)
plt.axhline(0, color='red', linestyle='--', linewidth=1, label='No net response (delta = 0)')
plt.title('AGE vs. IFN-Gamma Delta\n(Stimulated − Unstimulated) by BMI Category',
          fontsize=13, fontweight='bold', pad=12)
plt.xlabel('Age (years)', fontsize=12)
plt.ylabel('IFN-Gamma Delta (pg/mL)', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.7)
plt.legend(title='BMI Category', bbox_to_anchor=(1.01, 1), loc='upper left')
plt.tight_layout()
plt.savefig('../visuals/age_vs_ifn_gamma_delta_by_bmi.png', bbox_inches='tight', dpi=150)
plt.show()
print('Chart saved: ../visuals/age_vs_ifn_gamma_delta_by_bmi.png')

In [ ]:
# --- Visual 3: IFN-Gamma Delta Distribution by Outcome Group ---
# Shows how immune response strength differs across clinical trajectories
outcome_order = ['NON-CONVERTER', 'NON-PROGRESSOR', 'PROGRESSOR', 'REVERTER', 'CONVERTER']

plt.figure(figsize=(12, 6))
sns.boxplot(
    data=df_processed,
    x='OUTCOME',
    y='IFN_GAMMA_DELTA',
    order=outcome_order,
    palette='Set2'
)
plt.axhline(0, color='red', linestyle='--', linewidth=1.2, label='Baseline (delta = 0)')
plt.title('IFN-Gamma Delta (Stimulated − Unstimulated) by Outcome Group',
          fontsize=13, fontweight='bold', pad=12)
plt.xlabel('Outcome Group', fontsize=12)
plt.ylabel('IFN-Gamma Delta (pg/mL)', fontsize=12)
plt.xticks(rotation=10)
plt.legend()
plt.tight_layout()
plt.savefig('../visuals/ifn_gamma_delta_by_outcome.png', bbox_inches='tight', dpi=150)
plt.show()
print('Chart saved: ../visuals/ifn_gamma_delta_by_outcome.png')